# 第 26 章 · Computer Use：观察、动作与核验协议

一个 click 回执最多证明控制链承认了输入。业务是否成功，要重新观察目标状态，并把新观察和本次 action 关联起来。

本章**仅实现局部观察/动作账本**。url、revision 和 rows 全部由合成输入提供；调用 begin_dispatch 不会点击网页。没有打开浏览器、操控设备、采集真实截图或完成真实 task-list UI 验收。

**运行范围：局部机制实验。** 按顺序执行，每个实验都通过真实函数、文件或队列产生结果，再用断言检查。修改参数后重跑该实验，必要时先重启内核并执行全部。所有合成输入都只证明本地机制；本章没有累计检查点，真实服务与设备仍待验证。

本章使用原作者的设计与局部原型；下面明确写出的简化是教材选择，不是 Codex 原样实现。固定源码：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · 观察身份决定动作是否仍然有意义

观察包含 session、epoch、sequence、revision、origin。动作必须引用最近观察，并再次比较当前 revision；旧截图坐标不能永久有效。修改 current_revision 会在派发记录之前触发拒绝。

In [2]:
m = load_example('26', 'observation_ledger')
ledger = m.ObservationLedger('local-session', 'https://example.invalid')
before = ledger.observe(url='https://example.invalid/tasks', revision=1,
                        selected_status='all', rows=[('a', 'done'), ('b', 'todo')])
try:
    ledger.begin_dispatch('stale', before, current_url='https://example.invalid/tasks',
                          current_revision=2, granted=True)
except m.Rejected as error:
    assert str(error) == 'stale_or_wrong_target'
else:
    raise AssertionError('过期观察未拒绝')
attempt = ledger.begin_dispatch('action-1', before, current_url='https://example.invalid/tasks',
                                current_revision=1, granted=True)
assert attempt.phase == 'dispatched' and attempt.verified is None
print({'recorded_intent': attempt.action_id, 'actual_browser_input_sent': False})

{'recorded_intent': 'action-1', 'actual_browser_input_sent': False}


## 2 · 回执不能代替目标核验

先提供 acknowledged，再提供一个没有达到筛选目标的新观察。第二次输入改成真正匹配的行集合，观察 gate 改变。空列表不能靠 all([]) 偷过预期非空任务集。

In [3]:
assert ledger.receipt('action-1', 'acknowledged')
assert attempt.verified is None
wrong = ledger.observe(url='https://example.invalid/tasks', revision=2,
                       selected_status='done', rows=[])
assert ledger.verify_status_filter('action-1', wrong, status='done', expected_ids={'a'}) == 'goal_not_observed'
correct = ledger.observe(url='https://example.invalid/tasks', revision=3,
                         selected_status='done', rows=[('a', 'done')])
assert ledger.verify_status_filter('action-1', correct, status='done', expected_ids={'a'}) == 'goal_observed'
print({'receipt': attempt.receipt, 'controlled_input_verdict': attempt.verified, 'live_ui_verified': False})

{'receipt': 'acknowledged', 'controlled_input_verdict': 'goal_observed', 'live_ui_verified': False}


## 3 · origin 和授权是分开的入口检查

同一 host 的 http/https 或端口不同，属于不同 origin。跨域观察会废弃旧 epoch；重新回到原站点，也不能复用跨域前的快照。配置允许某个 origin 不等于每个动作均已批准。

In [4]:
old_epoch = ledger.epoch
try:
    ledger.observe(url='https://other.invalid/tasks', revision=4, selected_status='all', rows=[])
except m.Rejected as error:
    assert str(error) == 'origin_denied'
else:
    raise AssertionError('跨域应拒绝')
assert ledger.latest is None and ledger.epoch == old_epoch + 1
fresh = ledger.observe(url='https://example.invalid/tasks', revision=5, selected_status='all', rows=[])
try:
    ledger.begin_dispatch('unapproved', fresh, current_url='https://example.invalid/tasks',
                          current_revision=5, granted=False)
except m.Rejected as error:
    assert str(error) == 'permission_denied'
else:
    raise AssertionError('origin 许可不能代替动作批准')
assert m.exact_origin('https://example.invalid') != m.exact_origin('https://example.invalid:444')
print({'epoch_after_navigation': ledger.epoch, 'permission': 'denied'})

{'epoch_after_navigation': 2, 'permission': 'denied'}


## 4 · 取消保留未知副作用；重连要求新观察

取消后仍可能观察到目标已变化。账本保留 cancelled 控制事实，同时记录目标观察，不擅自回滚或把晚到回执当复活。重新打开 session 后，旧观察即使内容相同也失效。

In [5]:
cancel_before = ledger.observe(url='https://example.invalid/tasks', revision=6,
                               selected_status='all', rows=[('a', 'done')])
cancelled = ledger.begin_dispatch('action-2', cancel_before, current_url='https://example.invalid/tasks',
                                  current_revision=6, granted=True)
ledger.cancel('action-2')
assert not ledger.receipt('action-2', 'acknowledged')
after_cancel = ledger.observe(url='https://example.invalid/tasks', revision=7,
                             selected_status='done', rows=[('a', 'done')])
assert ledger.verify_status_filter('action-2', after_cancel, status='done', expected_ids={'a'}) == 'goal_observed'
assert cancelled.phase == 'cancelled_outcome_unknown' and cancelled.cancelled
ledger.close()
ledger.reopen()
try:
    ledger.begin_dispatch('action-3', after_cancel, current_url='https://example.invalid/tasks',
                          current_revision=7, granted=True)
except m.Rejected as error:
    assert str(error) == 'stale_or_wrong_target'
else:
    raise AssertionError('重连后旧观察应失效')
print({'control_state': cancelled.phase, 'observed_goal': cancelled.verified, 'reopened_epoch': ledger.epoch})

{'control_state': 'cancelled_outcome_unknown', 'observed_goal': 'goal_observed', 'reopened_epoch': 3}


## 源码对照与接回主线

以下为固定提交的已定位源码入口。阅读源码能解释设计边界，不等于执行过其测试；本章 Python 原型与这些 Rust 接口不是同一实现。

- [codex-rs/protocol/src/mcp.rs#L41-L52](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/protocol/src/mcp.rs#L41-L52)
- [codex-rs/config/src/browser_use.rs#L9-L22](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/config/src/browser_use.rs#L9-L22)
- [codex-rs/config/src/browser_use_tests.rs#L7-L51](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/config/src/browser_use_tests.rs#L7-L51)

**接回条件：** 将局部机制接入同一 Dispatcher、Scheduler、History 和受保护验证器，再执行原合同。目录、断言和演示输出都不能替代累计集成证据。